# Final Visualizations — Enterprise Multi-Modal Data Visualization

**Course:** Database Knowledge Extraction
**Project:** Enterprise Multi-Modal Data Visualization and Storytelling
**Owner:** Laura (Role 3 — Python Visualization Specialist)
**Status:** Scaffold — run every cell after Role 1 delivers `data/processed/clean_dataset.csv`.

## Objectives

This notebook covers the programmatic visual taxonomy required by the rubric:

1. **Distributions & spread** — histogram + KDE and box/violin plots.
2. **Uncertainty & variability** — monthly trend with a 95% confidence interval band.
3. **X–Y relationships** — Seaborn correlation heatmap with a diverging palette centered at 0.
4. **Interactive web component** — Plotly scatter/bubble plot with hover, zoom, and color.
5. **Export** — high-resolution PNGs and a standalone interactive HTML file.

> **QA rule (Role 5):** the final version must run top-to-bottom with
> `Kernel → Restart & Run All` and produce every asset listed in section 5.

## 0. Environment setup

Run from the repository root or from the `notebooks/` folder. Paths are resolved relative to
the project root, so no absolute paths are required.

```bash
pip install -r requirements.txt
jupyter notebook notebooks/final_visualizations.ipynb
```

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Resolve the project root whether the notebook is launched from the root or notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "clean_dataset.csv"
PLOTS_DIR = PROJECT_ROOT / "assets" / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Declutter every Matplotlib/Seaborn chart by default.
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
})

print(f"Project root: {PROJECT_ROOT}")
print(f"Data file:    {DATA_PATH}")
print(f"Plots folder: {PLOTS_DIR}")

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["Order Date", "Ship Date"])

print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
print(f"Date range: {df['Order Date'].min():%Y-%m-%d} -> {df['Order Date'].max():%Y-%m-%d}")
df.head()

In [ ]:
# Quick data-quality sanity check before plotting.
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique": df.nunique(),
})
quality.sort_values("missing", ascending=False)

## 1. Distributions & spread

**Question:** How are revenue and profit distributed, and where are the outliers?

- `Sales` is typically right-skewed: a few high-value orders dominate revenue.
- `Profit` contains both positive and negative values; the box plot exposes loss-making
  categories and extreme outliers.

**Design choices:** 50 bins for the histogram, KDE overlay for shape, box plot for spread and
outliers. Top/right spines removed, no redundant legend.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(
    data=df,
    x="Sales",
    bins=50,
    kde=True,
    color="#4C72B0",
    edgecolor="white",
    ax=axes[0],
)
axes[0].set_title("Distribution of Sales")
axes[0].set_xlabel("Sales (USD)")
axes[0].set_ylabel("Order lines")
axes[0].axvline(df["Sales"].median(), color="#C44E52", linestyle="--", linewidth=1.5,
                label=f"Median: ${df['Sales'].median():,.0f}")
axes[0].legend(frameon=False)

sns.boxplot(
    data=df,
    x="Category",
    y="Profit",
    color="#4C72B0",
    width=0.55,
    fliersize=2,
    ax=axes[1],
)
axes[1].axhline(0, color="#333333", linewidth=1)
axes[1].set_title("Profit Spread by Category")
axes[1].set_xlabel("")
axes[1].set_ylabel("Profit (USD)")

plt.tight_layout()
plt.savefig(PLOTS_DIR / "distribution_kde.png", bbox_inches="tight")
plt.show()

**Interpretation (complete after running):**

- Skewness of `Sales`: TBD
- Median profit: TBD
- Category with the widest spread: TBD
- Share of loss-making order lines: TBD

## 2. Uncertainty & variability

**Question:** How confident are we in the monthly profit trend?

The shaded band is a **95% confidence interval** for the mean monthly profit
(`errorbar=("ci", 95)`). A wider band means more uncertainty; overlapping bands across months
mean the difference may not be meaningful.

**Design choices:** zero reference line, marker per month, no top/right spines.

In [ ]:
# Convert the period string to a timestamp so the x-axis is ordered correctly.
df["Order Month Date"] = df["Order Date"].dt.to_period("M").dt.to_timestamp()

fig, ax = plt.subplots(figsize=(15, 5))
sns.lineplot(
    data=df,
    x="Order Month Date",
    y="Profit",
    errorbar=("ci", 95),
    n_boot=1000,
    marker="o",
    markersize=4,
    linewidth=2,
    color="#4C72B0",
    ax=ax,
)
ax.axhline(0, color="#333333", linewidth=1)
ax.set_title("Monthly Profit with 95% Confidence Interval")
ax.set_xlabel("")
ax.set_ylabel("Mean profit (USD)")
ax.margins(x=0.01)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "uncertainty_ci.png", bbox_inches="tight")
plt.show()

**Interpretation (complete after running):**

- Months with a confidence band entirely below zero: TBD
- Widest uncertainty: TBD
- Overall direction of the trend: TBD

## 3. X–Y relationships & correlation heatmap

**Question:** Which numeric variables move together, and which move in opposite directions?

The heatmap uses a **diverging palette centered at 0** (`vlag`): blue for positive
correlation, red for negative. The upper triangle is masked to avoid duplicate information.

> Correlation is **association, not causation**. Discount and profit may be related because
> both depend on category and pricing policy.

In [ ]:
numeric_columns = [
    "Sales",
    "Quantity",
    "Discount",
    "Profit",
    "Shipping Days",
    "Profit Margin",
]
available_numeric = [column for column in numeric_columns if column in df.columns]
corr = df[available_numeric].corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.6,
    cbar_kws={"shrink": 0.8, "label": "Pearson correlation"},
    ax=ax,
)
ax.set_title("Correlation Matrix — Numeric Variables")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "correlation_heatmap.png", bbox_inches="tight")
plt.show()

**Interpretation (complete after running):**

- Strongest positive relationship: TBD
- Strongest negative relationship: TBD
- Is `Discount` negatively associated with `Profit`? TBD

## 4. Interactive web component

**Question:** Which discount levels are associated with losses, and does the pattern differ by
category?

This Plotly bubble chart supports hover tooltips, zoom, pan, and legend filtering. Bubble size
encodes `Sales`; color encodes `Category` with a colorblind-safe qualitative palette.

The output is a standalone HTML file (`include_plotlyjs="cdn"`) that can be embedded in the
presentation or opened directly in a browser.

In [ ]:
fig = px.scatter(
    df,
    x="Discount",
    y="Profit",
    color="Category",
    size="Sales",
    size_max=28,
    opacity=0.6,
    hover_data={
        "Order ID": True,
        "Product Name": True,
        "Sub-Category": True,
        "Region": True,
        "Sales": ":.2f",
        "Profit": ":.2f",
    },
    color_discrete_sequence=px.colors.qualitative.Safe,
    title="Discount vs. Profit by Category (bubble size = Sales)",
    labels={"Discount": "Discount rate", "Profit": "Profit (USD)"},
)
fig.add_hline(y=0, line_dash="dash", line_color="#555555")
fig.update_layout(
    template="plotly_white",
    legend_title_text="Category",
    hovermode="closest",
    margin=dict(l=40, r=20, t=60, b=40),
)
fig.write_html(PLOTS_DIR / "interactive_scatter.html", include_plotlyjs="cdn")
fig.show()

## 5. Export verification

Every asset below is required by the project brief. The final notebook must show **OK** for all
five files.

In [ ]:
expected_assets = [
    "distribution_kde.png",
    "correlation_heatmap.png",
    "uncertainty_ci.png",
    "interactive_scatter.html",
]

for name in expected_assets:
    path = PLOTS_DIR / name
    status = "OK" if path.exists() else "MISSING"
    size_kb = path.stat().st_size / 1024 if path.exists() else 0
    print(f"{status:<8} {name:<30} {size_kb:,.1f} KB")

## 6. Findings for the executive story

Complete these bullets and send them to Josmar (Role 4) for the deck. Keep each insight
one sentence and tie it to a recommendation.

1. **Revenue:** TBD — what the sales distribution reveals.
2. **Profitability:** TBD — which categories/sub-categories destroy margin.
3. **Risk:** TBD — the discount threshold associated with losses.
4. **Opportunity:** TBD — the segment/region with the best margin.
5. **Recommendation:** TBD — the single most important action.

### Handoff checklist

- [ ] Notebook executed from top to bottom (`Kernel → Restart & Run All`).
- [ ] Markdown interpretation filled in for sections 1–3.
- [ ] All four assets exported to `assets/plots/`.
- [ ] Findings sent to Role 4 (Josmar) and Role 5 (Jesus).
- [ ] No absolute local paths in the notebook.